# Week 35, Day 3: Factory

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w035d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import json
from datetime import date

class Circle:
    def __init__(self, r):
        self.r = r
    def area(self):
        return round(3.14159 * self.r ** 2, 1)

class Rect:
    def __init__(self, w, h):
        self.w, self.h = w, h
    def area(self):
        return self.w * self.h

SHAPES = {"circle": Circle, "rect": Rect}

specs = json.loads('[{"kind": "rect", "w": 2, "h": 3}, {"kind": "circle", "r": 1}]')
shapes = [SHAPES[s.pop("kind")](**s) for s in specs]
print([type(s).__name__ for s in shapes], [s.area() for s in shapes])
print(date.fromisoformat("2026-11-03"), dict.fromkeys("ab", 0))

In [ ]:
class Circle:
    def __init__(self, r):
        self.r = r

class Rect:
    def __init__(self, w, h):
        self.w, self.h = w, h

class Square(Rect):
    def __init__(self, side):
        super().__init__(side, side)

SHAPES = {"circle": Circle, "rect": Rect, "square": Square}

def make_shape(spec):
    """Make a shape from a dictionary like {"kind": "circle", "r": 1}."""
    spec = dict(spec)                                # don't change the caller's dictionary
    kind = spec.pop("kind", None)
    if kind not in SHAPES:
        raise ValueError(f"unknown kind {kind!r}; choose from {', '.join(sorted(SHAPES))}")
    try:
        return SHAPES[kind](**spec)
    except TypeError as err:
        raise ValueError(f"bad settings for {kind}: {err}") from None

print(type(make_shape({"kind": "square", "side": 4})).__name__, make_shape({"kind": "rect", "w": 2, "h": 5}).w)
for bad in [{"kind": "hexagon"}, {"kind": "circle", "radius": 1}]:
    try:
        make_shape(bad)
    except ValueError as err:
        print("ValueError:", err)

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, f):
        return cls(round((f - 32) * 5 / 9, 1))

    @classmethod
    def from_text(cls, text):
        text = text.strip().upper()
        if text.endswith("F"):
            return cls.from_fahrenheit(float(text[:-1]))
        return cls(float(text.rstrip("C")))

    def __repr__(self):
        return f"{type(self).__name__}({self.celsius})"

class OvenTemperature(Temperature):
    pass

print(Temperature.from_text("21.5C"), Temperature.from_text(" 98.6f "), OvenTemperature.from_text("356F"))

In [ ]:
class Exporter:
    formats = {}

    def __init_subclass__(cls, name=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if name is not None:
            if name in Exporter.formats:
                raise ValueError(f"format {name!r} is already registered")
            Exporter.formats[name] = cls

    @classmethod
    def create(cls, name):
        if name not in cls.formats:
            raise ValueError(f"unknown format {name!r}")
        return cls.formats[name]()

    def export(self, rows):
        raise NotImplementedError

class CsvExporter(Exporter, name="csv"):
    def export(self, rows):
        return "\n".join(",".join(map(str, row)) for row in rows)

class TsvExporter(Exporter, name="tsv"):
    def export(self, rows):
        return "\n".join("\t".join(map(str, row)) for row in rows)

print(sorted(Exporter.formats))
print(Exporter.create("csv").export([["rice", 450], ["oil", 900]]))

In [ ]:
from dataclasses import dataclass

@dataclass
class Button:
    label: str
    colour: str

@dataclass
class Panel:
    colour: str

class LightTheme:
    def button(self, label):
        return Button(label, "#ffffff")
    def panel(self):
        return Panel("#f2f2f2")

class DarkTheme:
    def button(self, label):
        return Button(label, "#222222")
    def panel(self):
        return Panel("#111111")

def build_screen(theme):
    return [theme.panel(), theme.button("OK"), theme.button("Cancel")]

for theme in (LightTheme(), DarkTheme()):
    print([getattr(w, "colour") for w in build_screen(theme)])

> **Key idea.** A factory is any function, method or object whose job is creating objects, so the decision about which class to create lives in one place. Factory functions use a registry (a dictionary from names to classes) and turn bad descriptions into clear errors. Alternative constructors are class methods that call `cls(...)`, so they work for subclasses; the Factory Method pattern lets subclasses override a creation method. `__init_subclass__` lets subclasses register themselves when defined, keyword arguments in the class line included. An Abstract Factory makes a whole family of matching objects, swapped as one.

### Exercise: A factory function

Write `make_vehicle(spec)`, which turns a dictionary like `{"type": "bike", "code": "B1"}` into an object, using a registry `VEHICLES` of the classes given. The `"type"` key chooses the class, and the other keys are passed to it as keyword arguments. An unknown type, and settings the class doesn't accept (a `TypeError` from the class), both raise `ValueError`. The caller's dictionary must not be changed.

In [ ]:
class Bike:
    def __init__(self, code):
        self.code = code

class EBike(Bike):
    def __init__(self, code, battery=100):
        super().__init__(code)
        self.battery = battery

class Scooter:
    def __init__(self, code, battery=100):
        self.code, self.battery = code, battery

___

def make_vehicle(spec):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_factory():\n    \"making vehicles\"\n    B, E, S = ns[\"Bike\"], ns[\"EBike\"], ns[\"Scooter\"]\n    spec = {\"type\": \"ebike\", \"code\": \"E1\", \"battery\": 40}\n    v = make_vehicle(spec)\n    assert type(v) is E and v.battery == 40 and spec == {\"type\": \"ebike\", \"code\": \"E1\", \"battery\": 40}\n    assert type(make_vehicle({\"type\": \"bike\", \"code\": \"B1\"})) is B and make_vehicle({\"type\": \"scooter\", \"code\": \"S1\"}).battery == 100\n    for bad in [{\"type\": \"car\", \"code\": \"C1\"}, {\"code\": \"X\"}, {\"type\": \"bike\", \"code\": \"B1\", \"battery\": 5}, {\"type\": \"bike\"}]:\n        try:\n            make_vehicle(bad)\n            assert False, f\"{bad} should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: Alternative constructors

Give `Money` three class methods that all use `cls(...)`: `from_pounds(pounds)` (a float, rounded to the nearest penny), `from_text(text)` (`"£12.50"`, `"12.50"` or `"75p"`, spaces at the ends allowed; anything else raises `ValueError`), and `zero()`. A subclass `Fee` must get `Fee` objects from all three.

In [ ]:
import re

class Money:
    def __init__(self, pence):
        self.pence = pence

    def __repr__(self):
        return f"{type(self).__name__}({self.pence})"

    ___

class Fee(Money):
    pass

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ctors():\n    \"constructors\"\n    M, F = ns[\"Money\"], ns[\"Fee\"]\n    assert repr(M.from_pounds(12.5)) == \"Money(1250)\" and repr(M.from_pounds(0.1 + 0.2)) == \"Money(30)\"\n    assert [M.from_text(t).pence for t in [\"\u00a312.50\", \" 12.50 \", \"75p\", \"\u00a33\", \"\u00a30.5\"]] == [1250, 1250, 75, 300, 50]\n    assert repr(F.from_text(\"\u00a31\")) == \"Fee(100)\" and type(F.zero()) is F and type(F.from_pounds(1.0)) is F and M.zero().pence == 0\n    for bad in [\"12,50\", \"\u00a3\", \"p\", \"\u00a31.234\", \"ten\"]:\n        try:\n            M.from_text(bad)\n            assert False, f\"{bad!r} should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: Commands that register themselves

Write a base class `Command` whose subclasses register themselves with `__init_subclass__`, using a `name` keyword in the class line (`class Add(Command, name="add")`). Subclasses without a name aren't registered; a duplicate name raises `ValueError` when the class is defined. `Command.run(line)` splits the line, makes an object of the command named by the first word, and returns its `execute(args)` result (with the remaining words as a list); an unknown command gives the text `unknown command: xyz`. Then write `Add` (`add 2 3 4` gives `9`, as an `int`) and `Shout` (`shout hi there` gives `HI THERE`).

In [ ]:
class Command:
    registry = {}
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_commands():\n    \"self-registering\"\n    C = ns[\"Command\"]\n    assert C.run(\"add 2 3 4\") == 9 and C.run(\"shout hi there\") == \"HI THERE\" and C.run(\"dance now\") == \"unknown command: dance\"\n    assert sorted(C.registry) == [\"add\", \"shout\"]\n    class Count(C, name=\"count\"):\n        def execute(self, args):\n            return len(args)\n    assert C.run(\"count a b c\") == 3\n    class Helper(C):\n        pass\n    assert \"Helper\" not in [c.__name__ for c in C.registry.values()]\n    try:\n        class Again(C, name=\"add\"):\n            pass\n        assert False, \"a duplicate name should raise ValueError\"\n    except ValueError:\n        pass\n    del C.registry[\"count\"]")

### Exercise: A reader for each file type

Write `reader_for(path)`, a factory that looks at a file name's extension (any case) and returns a **function** that reads that file and returns a list of rows (each row a list of strings): `.csv` files are read with `csv.reader`; `.tsv` files are split on tabs; `.txt` files give one row per non-blank line, with the line as its only item. Use a dictionary `READERS` from extension to function, and `pathlib.Path(path).suffix`. An unknown extension raises `ValueError`. Then write `load(path)`, which uses the factory.

In [ ]:
# Run this cell first: it creates the files used below.
with open('stock.csv', 'w', encoding='utf-8') as f:
    f.write('rice,450\noil,900\n')
with open('stock.tsv', 'w', encoding='utf-8') as f:
    f.write('rice\t450\n')
with open('notes.TXT', 'w', encoding='utf-8') as f:
    f.write('buy rice\n\ncall Kemi\n')

In [ ]:
import csv
from pathlib import Path

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_readers():\n    \"the right reader\"\n    assert load(\"stock.csv\") == [[\"rice\", \"450\"], [\"oil\", \"900\"]] and load(\"stock.tsv\") == [[\"rice\", \"450\"]]\n    assert load(\"notes.TXT\") == [[\"buy rice\"], [\"call Kemi\"]] and callable(reader_for(\"x.csv\"))\n    assert set(READERS) >= {\".csv\", \".tsv\", \".txt\"}\n    for bad in [\"data.json\", \"README\"]:\n        try:\n            reader_for(bad)\n            assert False, f\"{bad} should raise ValueError\"\n        except ValueError:\n            pass")

### Bug hunt: A class, not an object

The factory hands back the class itself instead of an object of it, so this crashes with `TypeError: Circle.area() missing 1 required positional argument: 'self'`. Fix `make_shape` so it prints `12.57`.

In [ ]:
class Circle:
    def __init__(self, r):
        self.r = r

    def area(self):
        return round(3.14159265 * self.r ** 2, 2)

SHAPES = {"circle": Circle}

def make_shape(kind, **settings):
    return SHAPES[kind]

print(make_shape("circle", r=2).area())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"an object\"\n    expect_output(\"12.57\")")

### Bug hunt: Upper case in, lower case out

Formats are registered in upper case but asked for in whatever case the user types, so `"csv"` isn't found and this prints `unknown format csv`. Make the registry case-insensitive by normalising names to lower case **both** when registering and when looking up, so it prints `CSV export` and then `JSON export`.

In [ ]:
FORMATS = {}

def register(name, func):
    FORMATS[name] = func

def export(name):
    if name not in FORMATS:
        return f"unknown format {name}"
    return FORMATS[name]()

register("CSV", lambda: "CSV export")
register("Json", lambda: "JSON export")
print(export("csv"))
print(export("JSON"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"any case\"\n    expect_output(\"CSV export\\nJSON export\")\n    assert sorted(FORMATS) == [\"csv\", \"json\"]")

### Bug hunt: One connection per place

`connection(name)` should give **one** shared connection per database name, made the first time it's asked for, but it makes a new one every call: this prints `3` instead of `2`. Fix the factory so it keeps the connections it has made in `_CONNECTIONS` and hands back the same one for the same name.

In [ ]:
class Connection:
    opened = 0

    def __init__(self, name):
        self.name = name
        Connection.opened += 1

_CONNECTIONS = {}

def connection(name):
    return Connection(name)

a = connection("shop")
b = connection("shop")
c = connection("stock")
print(Connection.opened)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"shared\"\n    expect_output(\"2\")\n    assert connection(\"shop\") is connection(\"shop\") and connection(\"shop\") is not connection(\"stock\")")

### Puzzle: A self-registering family

Arrange the lines so that each `Unit` subclass registers itself under its symbol. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class Base:
    made = []
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Base.made.append(cls.__name__)

class A(Base):
    @classmethod
    def build(cls):
        return cls()

class B(A):
    pass

KINDS = {"a": A, "b": B}

p = ___   # Base.made
q = ___   # type(B.build()).__name__
r = ___   # type(KINDS["a"]()).__name__
s = ___   # KINDS["b"] is B
t = ___   # isinstance(KINDS["b"](), A)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [\"A\", \"B\"] and (q, r) == (\"B\", \"A\"), \"p to r\"\n    assert s is True and t is True")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a pipeline built from settings

A text-cleaning pipeline is described in JSON, so that people can change it without touching code:

```json
[{"step": "strip"}, {"step": "replace", "old": "&", "new": "and"}, {"step": "truncate", "length": 20}]
```

Build it with factories:

- `Step`, a base class whose subclasses register themselves with `__init_subclass__` and a `name` keyword. Each step is callable: `step(text)` returns the changed text.
- Steps: `Strip` (name `"strip"`, no settings), `Lower` (`"lower"`), `Replace` (`"replace"`, settings `old` and `new`), and `Truncate` (`"truncate"`, setting `length`, at least 1; if it cuts anything, the result is the first `length - 1` characters and `"…"`).
- `Step.from_spec(spec)`: a factory class method making a step from one dictionary; an unknown step name, a missing `"step"` key, or wrong settings raise `ValueError`.
- `Pipeline(steps)`, callable, applying the steps in order, with a class method `from_json(text)` that builds every step with `Step.from_spec`. Any problem raises `ValueError` whose message starts with the step's position, like `step 2: unknown step 'shout'` (counting from 0).

Adding a new kind of step must only need a new subclass.

In [ ]:
import json

class Step:
    registry = {}
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("CONFIG = '[{\"step\": \"strip\"}, {\"step\": \"replace\", \"old\": \"&\", \"new\": \"and\"}, {\"step\": \"truncate\", \"length\": 20}]'\ndef test_steps():\n    \"steps\"\n    S = ns[\"Step\"]\n    assert sorted(S.registry) == [\"lower\", \"replace\", \"strip\", \"truncate\"]\n    assert S.from_spec({\"step\": \"strip\"})(\"  hi \") == \"hi\" and S.from_spec({\"step\": \"replace\", \"old\": \"a\", \"new\": \"o\"})(\"banana\") == \"bonono\"\n    assert S.from_spec({\"step\": \"truncate\", \"length\": 4})(\"Lagos\") == \"Lag\u2026\" and S.from_spec({\"step\": \"truncate\", \"length\": 5})(\"Lagos\") == \"Lagos\"\n    for bad in [{\"step\": \"shout\"}, {\"old\": \"a\"}, {\"step\": \"replace\", \"old\": \"a\"}, {\"step\": \"truncate\", \"length\": 0}, {\"step\": \"strip\", \"extra\": 1}]:\n        try:\n            S.from_spec(bad)\n            assert False, f\"{bad} should raise ValueError\"\n        except ValueError:\n            pass\ndef test_pipeline():\n    \"pipelines\"\n    P, S = ns[\"Pipeline\"], ns[\"Step\"]\n    p = P.from_json(CONFIG)\n    assert p(\"   Fish & chips & mushy peas   \") == \"Fish and chips and \u2026\" and p(\"  Tea  \") == \"Tea\"\n    try:\n        P.from_json('[{\"step\": \"strip\"}, {\"step\": \"lower\"}, {\"step\": \"shout\"}]')\n        assert False, \"should raise ValueError\"\n    except ValueError as err:\n        assert str(err).startswith(\"step 2:\"), f\"message: {err}\"\n    class Reverse(S, name=\"reverse\"):\n        def __call__(self, text):\n            return text[::-1]\n    try:\n        assert P.from_json('[{\"step\": \"reverse\"}, {\"step\": \"lower\"}]')(\"ABC\") == \"cba\"\n    finally:\n        del S.registry[\"reverse\"]")

## Warm-up

The shapes were described in JSON, and one line turned each description into an object of the right class. The last line shows two factories in Python's own library: `date.fromisoformat` and `dict.fromkeys` make objects from something other than the usual arguments. Today is about writing code like both.

## Who decides what to make?

Creating an object means naming its class, and naming a class couples your code to it (Week 34). That's fine for most code, but when **which** class to create depends on data, the decision tends to be copied:

```python
if spec["kind"] == "circle":
    shape = Circle(spec["r"])
elif spec["kind"] == "rect":
    shape = Rect(spec["w"], spec["h"])
```

If this appears in the loader, the editor and the import tool, each new shape means three edits, and they can drift apart. A **factory** is any function, method or object whose job is to create objects, so that the decision lives in **one** place. The Gang of Four describe two patterns (Factory Method and Abstract Factory); in everyday Python, "factory" usually means any of the forms below.

## Factory functions

A factory function takes a description and returns an object. A dictionary from names to classes (a **registry**) does the choosing, since classes are values like any other:

The factory also gives one place for **checks** and **good error messages**: here, an unknown kind and wrong settings both become a clear `ValueError` instead of a `KeyError` or a confusing `TypeError` from deep inside.

## Alternative constructors

A class's `__init__` takes one set of arguments. When objects can also be made from other things (text, a dictionary, a file), give the class **alternative constructors**: class methods that build the arguments and call `cls(...)`. You've written these since Week 28; here's why `cls` matters for factories:

Because the class methods call `cls(...)`, `OvenTemperature.from_text` makes an `OvenTemperature`. The Gang of Four's **Factory Method** pattern is the same idea taken further: a parent class calls a method to create an object it needs, and **subclasses override that method** to decide which class it is. For example, a `Report` might call `self.make_formatter()`, with `HtmlReport` and `TextReport` each returning their own kind of formatter.

## Classes that register themselves

A registry has a weakness: every new class must also be added to the dictionary, and forgetting is easy. Python can do the registering automatically. When a class is defined, Python calls its parent's `__init_subclass__` method with the new class; a parent can use that to record each child:

- `class CsvExporter(Exporter, name="csv")` passes `name` to `__init_subclass__`, alongside the parent class in the brackets.
- `__init_subclass__` is automatically a class method: `cls` is the **new subclass** being defined.
- Calling `super().__init_subclass__(**kwargs)` keeps it cooperative with other parents (Week 30, Day 3).
- The check for duplicates runs when the class is defined, so a clash fails at once, not when someone asks for that format.

Adding a format is now one class, in any module that gets imported. This is how many plugin systems work.

## Families of objects

Sometimes objects come in **families** that must match: a light theme's button, panel and text colour all belong together, and mixing a light button with a dark panel would look wrong. The Gang of Four's **Abstract Factory** is an object with a creation method for each member of the family; swapping the factory swaps the whole family at once:

`build_screen` never names a colour or a theme; it just asks the factory it was given (dependency inversion again). In Python, an abstract factory is often just an object or a module with a few functions, as here.

When is a factory worth it? When the class to create depends on **data**, when creating an object takes several steps that must be done the same way everywhere, or when callers shouldn't know the concrete classes at all. When one place creates one class with ordinary arguments, just call the class.

## Practice

## Quiz

## Daily challenge

## Before you go

Look for a place in your own code where an object's class is chosen by an `if` on a string, and decide whether a registry, an alternative constructor or self-registering classes would suit it best. Tomorrow: the Adapter pattern, for fitting things to the interface you need.